## 掛載雲端硬碟

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 更改檔案所在路徑

In [3]:
# Change to your own folder !!!
%cd /content/drive/MyDrive/EAI_Prunning

/content/drive/MyDrive/EAI_Prunning


## 載入函式庫

In [4]:
import os

import torch
import torch.nn as nn
from torch.autograd import Variable
from torchvision import datasets, transforms
import numpy as np

from resnet import ResNet50, Bottleneck

## 超參數設定

In [5]:
DATASET = 'cifar10'
TEST_BATCH_SIZE = 1000
CUDA = True
PRUNE_PERCENT = 0.9 # Change your prune ratio!
WEIGHT_PATH = '/content/drive/MyDrive/EAI_Prunning/model_best.pth' # Change to your own folder !!!
PRUNE_PATH = '/content/drive/MyDrive/EAI_Prunning/model_prune.pth' # Change to your own folder !!!

## 載入模型

In [6]:
CUDA = CUDA and torch.cuda.is_available()

model = ResNet50(num_classes=10)
if CUDA:
    model.cuda()

if WEIGHT_PATH:
    if os.path.isfile(WEIGHT_PATH):
        if CUDA:
            checkpoint = torch.load(WEIGHT_PATH)
        else:
            checkpoint = torch.load(WEIGHT_PATH, map_location=torch.device('cpu'))
        best_prec1 = checkpoint['best_prec1']
        model.load_state_dict(checkpoint['state_dict'])
        print('LOADING CHECKPOINT {} @EPOCH={}, BEST_PREC1={}'.format(WEIGHT_PATH,checkpoint['epoch'],best_prec1))
    else:
        print("NO CHECKPOINT FOUND")

print(model)

LOADING CHECKPOINT /content/drive/MyDrive/EAI_Prunning/model_best.pth @EPOCH=40, BEST_PREC1=0.9018999934196472
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): 

## 進行剪枝
#### 計算所有 Bottleneck 中 bn1 和 bn2 的 scale factor 絕對值大小並排序
#### 利用設定好的 PRUNE_PERCENT 來取得閾值
#### 注意:只剪枝 Bottleneck 內部的 bn1 和 bn2,不剪枝 bn3 和其他 BN

In [7]:
# 只統計 Bottleneck 內的 bn1 和 bn2
total = 0
for m in model.modules():
    if isinstance(m, Bottleneck):
        total += m.bn1.weight.data.shape[0]
        total += m.bn2.weight.data.shape[0]

bn = torch.zeros(total)
index = 0
for m in model.modules():
    if isinstance(m, Bottleneck):
        # 收集 bn1 的權重
        size = m.bn1.weight.data.shape[0]
        bn[index:(index+size)] = m.bn1.weight.data.abs().clone()
        index += size

        # 收集 bn2 的權重
        size = m.bn2.weight.data.shape[0]
        bn[index:(index+size)] = m.bn2.weight.data.abs().clone()
        index += size

y, i = torch.sort(bn)

# 處理邊界情況
if total > 0:
    threshold_index = int(total * PRUNE_PERCENT)
    if threshold_index == total:
        threshold_index = total - 1
    threshold = y[threshold_index]
else:
    threshold = 0
    print("警告: 找不到可剪枝的層 (Bottleneck 內的 bn1, bn2)。")

print(f"Total prunable BN parameters: {total}")
print(f"Threshold: {threshold}")

Total prunable BN parameters: 7552
Threshold: 0.3734133839607239


## 根據 Batch Normalization Layer 資訊建立 CONFIG
#### 1. 複製 Batch Normalization Layer 的 weight (也就是 scale factor γ)
#### 2. 建立 mask,大於 threshold 的 index 的值會設成 1,小於 threshold 的值會設成 0
#### 3. mask 的值加總後,會是剪枝後 Layer 對應的輸出 channel 數
#### 4. 最後得到要建立剪枝模型的 CONFIG
#### 注意:按照模型結構順序處理所有 BN 層

In [8]:
pruned = 0
cfg = []  # 用來建立剪枝網路的 CONFIG
cfg_mask = []  # 用來幫助剪枝的遮罩

In [9]:
# 按照模型結構順序處理
# 1. 處理 model.bn1 (不剪枝)
m = model.bn1
num_channels = m.weight.data.shape[0]
if CUDA:
    mask = torch.ones(num_channels).cuda()
else:
    mask = torch.ones(num_channels)
cfg.append(num_channels)
cfg_mask.append(mask.clone())
print(f"Layer (model.bn1) \t total channel: {num_channels} \t remaining channel: {num_channels} (Not Pruned)")

# 2. 處理所有 Bottleneck 中的 BN 層
for m in model.modules():
    if isinstance(m, Bottleneck):
        # 處理 bn1 (剪枝)
        weight_copy = m.bn1.weight.data.abs().clone()
        if CUDA:
            mask = weight_copy.gt(threshold).float().cuda()
        else:
            mask = weight_copy.gt(threshold).float()

        # 注意: 需自行設計處理剩下channel數為0的情況 (e.g. 至少保留3個channel)
        ########################################################################
        # 如果剪枝後通道數為 0,則至少保留 1 個通道
        if int(torch.sum(mask)) == 0:
            mask[torch.argmax(weight_copy)] = 1.0
        ########################################################################

        pruned += mask.shape[0] - torch.sum(mask)
        num_remaining = int(torch.sum(mask))
        cfg.append(num_remaining)
        cfg_mask.append(mask.clone())
        print(f"Layer (bn1) \t total channel: {mask.shape[0]} \t remaining channel: {num_remaining} (Pruned)")

        # ----------------------------------------------------------------------------------------

        # 處理 bn2 (剪枝)
        weight_copy = m.bn2.weight.data.abs().clone()
        if CUDA:
            mask = weight_copy.gt(threshold).float().cuda()
        else:
            mask = weight_copy.gt(threshold).float()


        # 注意: 需自行設計處理剩下channel數為0的情況 (e.g. 至少保留3個channel)
        ########################################################################
        # 如果剪枝後通道數為 0,則至少保留 1 個通道
        if int(torch.sum(mask)) == 0:
            mask[torch.argmax(weight_copy)] = 1.0
        ########################################################################

        pruned += mask.shape[0] - torch.sum(mask)
        num_remaining = int(torch.sum(mask))
        cfg.append(num_remaining)
        cfg_mask.append(mask.clone())
        print(f"Layer (bn2) \t total channel: {mask.shape[0]} \t remaining channel: {num_remaining} (Pruned)")

        # ------------------------------------------------------------------------------------------------------------

        # 處理 bn3 (不剪枝)
        num_channels = m.bn3.weight.data.shape[0]
        if CUDA:
            mask = torch.ones(num_channels).cuda()
        else:
            mask = torch.ones(num_channels)
        cfg.append(num_channels)
        cfg_mask.append(mask.clone())
        print(f"Layer (bn3) \t total channel: {num_channels} \t remaining channel: {num_channels} (Not Pruned)")

pruned_ratio = pruned / total if total > 0 else 0
print(f'\nTotal prunable channels (bn1, bn2): {total}')
print(f'Pruned channels: {int(pruned)}')
print(f'Pruned ratio: {pruned_ratio:.3f}')
print('PREPROCESSING SUCCESSFUL!')
print(f'\ncfg length: {len(cfg)}')
print(f'cfg: {cfg}')

Layer (model.bn1) 	 total channel: 64 	 remaining channel: 64 (Not Pruned)
Layer (bn1) 	 total channel: 64 	 remaining channel: 64 (Pruned)
Layer (bn2) 	 total channel: 64 	 remaining channel: 64 (Pruned)
Layer (bn3) 	 total channel: 256 	 remaining channel: 256 (Not Pruned)
Layer (bn1) 	 total channel: 64 	 remaining channel: 64 (Pruned)
Layer (bn2) 	 total channel: 64 	 remaining channel: 64 (Pruned)
Layer (bn3) 	 total channel: 256 	 remaining channel: 256 (Not Pruned)
Layer (bn1) 	 total channel: 64 	 remaining channel: 63 (Pruned)
Layer (bn2) 	 total channel: 64 	 remaining channel: 62 (Pruned)
Layer (bn3) 	 total channel: 256 	 remaining channel: 256 (Not Pruned)
Layer (bn1) 	 total channel: 128 	 remaining channel: 69 (Pruned)
Layer (bn2) 	 total channel: 128 	 remaining channel: 74 (Pruned)
Layer (bn3) 	 total channel: 512 	 remaining channel: 512 (Not Pruned)
Layer (bn1) 	 total channel: 128 	 remaining channel: 31 (Pruned)
Layer (bn2) 	 total channel: 128 	 remaining channel:

## 建立剪枝模型

In [10]:
newmodel = ResNet50(num_classes=10, cfg=cfg)
if CUDA:
    newmodel.cuda()
print(newmodel)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 

## 將原本的模型權重複製到剪枝的模型
#### 根據不同層決定要複製什麼權重
###### Batch Normalization Layer
1. scale factor
2. bias
3. running mean
4. running variance

###### Convolutional Layer
1. weight

###### Linear Layer
1. weight
2. bias

In [11]:
old_modules = list(model.modules())
new_modules = list(newmodel.modules())

layer_id_in_cfg = 0
if CUDA:
    start_mask = torch.ones(3).cuda()  # 3 為 input channel (R,G,B)
else:
    start_mask = torch.ones(3)
end_mask = cfg_mask[layer_id_in_cfg]

for layer_id in range(len(old_modules)):
    m0 = old_modules[layer_id]
    m1 = new_modules[layer_id]

    if isinstance(m0, nn.BatchNorm2d):
        # 將原模型中被剪掉的 channel 對應的權重清零
        m0.weight.data.mul_(end_mask)
        m0.bias.data.mul_(end_mask)

        ################################################
        # 找出遮罩中非零元素的 index
        index = np.squeeze(np.argwhere(np.asarray(end_mask.cpu().numpy())))
        if index.ndim == 0:
            index = np.expand_dims(index, 0)
        ################################################

        ################################################
        # 複製 weight, bias, running mean, and running variance
        m1.weight.data = m0.weight.data[index.tolist()].clone()
        m1.bias.data = m0.bias.data[index.tolist()].clone()
        m1.running_mean = m0.running_mean[index.tolist()].clone()
        m1.running_var = m0.running_var[index.tolist()].clone()
        ################################################

        layer_id_in_cfg += 1
        start_mask = end_mask.clone()

        # 最後一層連接層不做修改
        if layer_id_in_cfg < len(cfg_mask):
            end_mask = cfg_mask[layer_id_in_cfg]

    elif isinstance(m0, nn.Conv2d):
        if isinstance(old_modules[layer_id + 1], nn.BatchNorm2d):
            idx0 = np.squeeze(np.argwhere(np.asarray(start_mask.cpu().numpy())))
            idx1 = np.squeeze(np.argwhere(np.asarray(end_mask.cpu().numpy())))
            if idx0.ndim == 0:
                idx0 = np.expand_dims(idx0, 0)
            if idx1.ndim == 0:
                idx1 = np.expand_dims(idx1, 0)

            ################################################
            # 複製 weight
            w = m0.weight.data[:, idx0.tolist(), :, :].clone()  # 先刪掉不需要的輸入通道
            w = w[idx1.tolist(), :, :, :].clone()  # 再刪掉不需要的輸出通道
            m1.weight.data = w.clone()  # 將精簡後的卷積權重複製到新模型
            ################################################

        # downsample 層不用 prune
        else:
            m1.weight.data = m0.weight.data.clone()

    elif isinstance(m0, nn.Linear):
        idx0 = np.squeeze(np.argwhere(np.asarray(start_mask.cpu().numpy())))
        if idx0.ndim == 0:
            idx0 = np.expand_dims(idx0, 0)

        ################################################
        # 複製 weight
        m1.weight.data = m0.weight.data[:, idx0.tolist()].clone()
        ################################################

        # 複製 bias
        m1.bias.data = m0.bias.data.clone()

print("權重複製完成!")

權重複製完成!


## 測試函數

In [12]:
def test(model):
    kwargs = {'num_workers': 1, 'pin_memory': True} if CUDA else {}
    test_loader = torch.utils.data.DataLoader(
        datasets.CIFAR10('./data', train=False, download=True, transform=transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))])),
        batch_size=TEST_BATCH_SIZE, shuffle=True, **kwargs)

    model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            if CUDA:
                data, target = data.cuda(), target.cuda()
            data, target = Variable(data), Variable(target)
            output = model(data)
            pred = output.data.max(1, keepdim=True)[1]
            correct += pred.eq(target.data.view_as(pred)).cpu().sum()

    print('\nTest set: Accuracy: {}/{} ({:.1f}%)\n'.format(
        correct, len(test_loader.dataset), 100. * correct / len(test_loader.dataset)))
    return correct / float(len(test_loader.dataset))

## 儲存模型並印出結果,以及剪枝後的 test acc

In [13]:
torch.save({'cfg': cfg, 'state_dict': newmodel.state_dict()}, PRUNE_PATH)

print("剪枝後的模型結構:")
print(newmodel)
print("\n測試剪枝後的模型準確率:")
test(newmodel)

剪枝後的模型結構:
ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Conv2d(64, 256, kernel_size=(1, 1), s

tensor(0.1970)